# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankita0531/ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is **classification**.

I want to classify content pages based on whether they show signs of declining search performance. Classification fits because the outcome can be represented as two classes: declining and not declining.

The model output can be used as decision support to help prioritize which pages should be reviewed first.

In [13]:
!git clone https://github.com/ankita0531/ML.git


Cloning into 'ML'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 126 (delta 39), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (126/126), 1.87 MiB | 16.80 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [14]:
%cd /content/ML

/content/ML


In [15]:
!ls data/raw

content_refresh_anonymized.csv


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df))
print("Columns related to the target:",
      [c for c in df.columns if "trend" in c.lower() or "declin" in c.lower()])

Rows: 30000
Columns: 30000
Columns related to the target: ['trend_direction', 'trend_pct']


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is `is_declining_label`.

It is a binary label based on the observed `trend_direction`. A page is labelled as declining when its observed trend direction is "down".

This gives the model a measurable outcome to learn from. The `trend_direction` and `trend_pct` columns should not be used as input features because they directly define or reveal the target.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())

print("\nClass proportions:")
print(df["is_declining_label"].value_counts(normalize=True))

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Class proportions:
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My main success metric is **recall for the declining class**.

Recall measures how many of the pages that are actually declining are correctly identified by the model. This is important because missing a genuinely declining page could mean missing an opportunity to review or refresh its content.

Precision can be used as a secondary measure to understand how much review effort the predictions may create.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import recall_score

y = df["is_declining_label"]

print("Total pages:", len(y))
print("Declining pages:", y.sum())
print("Declining rate:", y.mean())

Total pages: 30000
Declining pages: 16262
Declining rate: 0.5420666666666667


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one content page for a client.

Each row represents one client-content pair, identified by `client_hash_id` and `content_hash_id`. The model will use page-level performance features to estimate whether that content page is declining.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_df = df[
    [
        "client_id",
        "content_id",
        "is_declining_label"
    ]
].copy()

print("Dataframe shape:", lane_df.shape)
print("Number of rows:", lane_df.shape[0])
print("Number of columns:", lane_df.shape[1])

lane_df.head()

Dataframe shape: (30000, 3)
Number of rows: 30000
Number of columns: 3


,client_id,content_id,is_declining_label
0,client_f369cb89fc,content_304f48230142,1
1,client_4e07408562,content_a1fb4e703a9e,1
2,client_7f2253d7e2,content_9aa793d4d895,1
3,client_19581e27de,content_331d6c4de07b,0
4,client_3fdba35f04,content_d99b7a2d90ca,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule would require choosing one or a few thresholds and applying them to every page.

The available data contains multiple performance-related signals, such as impressions, clicks, sessions, engagement, content age, and average position. Their combinations may differ across content pages and clients.

Machine learning can learn patterns across multiple features instead of relying on one fixed threshold. The model would therefore provide directional decision support for prioritizing pages, while human review would still be needed before making content changes.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_columns = [
    c for c in df.columns
    if c not in [
        "client_id",
        "content_id",
        "trend_direction",
        "trend_pct",
        "is_declining_label"
    ]
]

print("Number of available features:", len(feature_columns))
print("\nExample features:")
print(feature_columns[:10])

Number of available features: 40

Example features:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.